# TP1 — Classification SMS Spam
## Analyse Critique & Conclusion Scientifique

**Dataset :** SMS Spam Collection — 5 574 messages (86.6 % ham / 13.4 % spam)
**Objectif :** Construire et comparer des pipelines NLP pour la détection automatique de spam

---


---
# 3. Analyse Critique

## Introduction

Ce TP a permis de construire et comparer **15 pipelines de classification automatique** de SMS, en faisant varier quatre facteurs : le nettoyage du texte, le prétraitement linguistique, la méthode de vectorisation et le modèle de classification.

Le dataset présente un **déséquilibre important** : 86.6 % des messages sont des ham (normaux) contre seulement 13.4 % de spam. Dans ce contexte, l'**accuracy seule est trompeuse** — un modèle naïf qui prédit tout en "ham" obtiendrait déjà 86.6 % d'accuracy sans détecter un seul spam. C'est pourquoi le **F1-score** (qui combine précision et rappel) est la métrique de référence tout au long de cette analyse.

---

## 3.1 Impact du Nettoyage

### Résultats obtenus

Le tableau ci-dessous compare les performances du pipeline TF-IDF + SVM selon le type de nettoyage appliqué :

| Nettoyage | Accuracy | Precision | Recall | **F1-score** |
|-----------|----------|-----------|--------|-------------|
| **Minimal** | 0.9862 | 0.9763 | 0.9196 | **0.9471**  |
| Avancé (Stem) | 0.9845 | 0.9760 | 0.9063 | 0.9398 |
| Avancé (Lemma) | 0.9827 | 0.9710 | 0.8973 | 0.9327 |

### Analyse et interprétation

Le nettoyage minimal obtient les **meilleures performances** — résultat contre-intuitif qui mérite une explication approfondie.

**Pourquoi un nettoyage agressif nuit-il aux performances ?**

Les SMS spam ont des caractéristiques textuelles très particulières que le nettoyage peut effacer :

- **Les majuscules sont des indicateurs forts.** Les spams utilisent massivement les majuscules pour attirer l'attention : "FREE PRIZE", "WIN NOW", "URGENT". Convertir tout en minuscules supprime ce signal. Un ham écrit "I am going home" tandis qu'un spam écrit "YOU HAVE WON £1000" — la capitalisation est informative.

- **Les chiffres signalent le spam.** Les montants d'argent ("£1000", "100% free"), les numéros surtaxés ("0870...", "09..."), les codes promo — ces éléments numériques sont quasi-exclusifs aux spams. Les supprimer appauvrit la représentation.

- **La ponctuation agressive est caractéristique.** Les points d'exclamation répétés ("You've won!!!"), les guillemets promotionnels ("WIN a prize!") sont des marqueurs stylistiques du spam. Les supprimer réduit le pouvoir discriminant du modèle.

- **Les URLs indiquent le spam.** Presque absentes des ham, très présentes dans les spam ("click www.prize.com"). Même après nettoyage minimal, les restes de ces tokens (mots comme "www", "http", "click") restent dans le texte et sont utiles.

**Ce que les matrices de confusion confirment :**

| Prétraitement | Vrais Ham (TN) | Faux Spam (FP) | Vrais Spam (TP) | Spams Manqués (FN) |
|---|---|---|---|---|
| **Minimal** | 1443 | **5** | **206** | **18** |
| Avancé Stem | 1443 | 5 | 203 | 21 |
| Avancé Lemma | 1442 | 6 | 201 | 23 |

Avec le nettoyage minimal : seulement **18 spams passent inaperçus** et 5 ham sont bloqués à tort. Le stemming fait monter à 21 spams manqués, la lemmatisation à 23. Le nettoyage agressif dégrade surtout le **rappel** — il laisse passer plus de spams.

> **Conclusion nettoyage :** Pour les SMS, le nettoyage doit rester modéré. Contrairement aux textes longs (articles, emails professionnels) où le bruit textuel est problématique, les SMS spam sont courts et leurs signaux textuels bruts (majuscules, chiffres, ponctuation) sont précisément ce qui les distingue des ham. Un nettoyage trop agressif supprime l'information discriminante avant même que le modèle puisse l'exploiter.

---

## 3.2 Impact du Prétraitement Linguistique

### Résultats obtenus

Le prétraitement avancé (stemming et lemmatisation) a été comparé au traitement minimal sur le même pipeline TF-IDF + SVM :

| Prétraitement | F1-score | Spams manqués (FN) |
|---|---|---|
| **Minimal** (tokenisation) | **0.9471** | **18** |
| Stemming (Porter) | 0.9398 | 21 |
| Lemmatisation (WordNet) | 0.9327 | 23 |

### Différence entre Stemming et Lemmatisation

Ces deux techniques normalisent les mots mais de façon différente :

**Le Stemming (Porter)** réduit les mots à leur racine brute par troncature de suffixes. C'est une opération purement mécanique, sans dictionnaire — il produit parfois des formes non réelles : "crazy" → "crazi", "available" → "avail", "goes" → "goe". C'est rapide et réduit le vocabulaire, mais au prix d'une certaine distorsion linguistique.

**La Lemmatisation (WordNet)** ramène les mots à leur forme canonique réelle en consultant un dictionnaire linguistique : "goes" → "go", "lives" → "life", "went" → "go". C'est plus précis et conserve un vocabulaire compréhensible, mais légèrement plus lent.

**Sur nos résultats :** La lemmatisation est linguistiquement plus correcte mais donne un F1 légèrement inférieur au stemming (0.9327 vs 0.9398). Les deux sont inférieurs au traitement minimal. Ce résultat s'explique ainsi :

**Pourquoi la normalisation morphologique dégrade-t-elle les performances ici ?**

1. **Les mots-clés spam sont déjà à leur forme simple.** "free", "win", "prize", "call", "cash" — ces mots apparaissent dans les spams sous leur forme de base. Stemmer ou lemmatiser n'apporte aucun regroupement supplémentaire utile.

2. **Les SMS sont très courts.** Avec 15 à 20 mots en moyenne, la diversité morphologique est faible. Il n'y a pas beaucoup de variantes à regrouper.

3. **La suppression des stopwords est risquée.** Les mots "you", "your", "have", "won", "free", "now" sont classés comme stopwords dans les listes standards — mais ce sont des indicateurs forts du spam. "You have won", "Your prize is ready", "Call now" : ces constructions typiques du spam disparaissent partiellement si on retire les stopwords.

4. **TF-IDF gère déjà le problème des mots fréquents.** La pondération IDF réduit automatiquement l'importance des mots très courants. La suppression manuelle des stopwords fait donc double emploi et risque d'éliminer des signaux utiles.

> **Conclusion prétraitement :** Le prétraitement avancé n'améliore pas les performances sur ce dataset. Il est plus adapté à de longs textes complexes (articles scientifiques, documents juridiques) où la normalisation morphologique aide à regrouper des variantes pertinentes. Sur des SMS courts au vocabulaire simple, le traitement minimal est optimal.

---

## 3.3 Impact de la Vectorisation

### Résultats obtenus

C'est le facteur ayant le **plus grand impact sur les performances** :

| Vectorisation | Meilleur F1 | Modèle | Rang |
|---|---|---|---|
| **TF-IDF** | **0.9368** | SVM | 1er |
| BoW | 0.9343 | SVM | 2ème |
| FastText | 0.8138 | SVM | 7ème |
| GloVe | 0.8000 | RF | 9ème |
| Word2Vec | 0.7789 | RF | 10ème |
| **Word2Vec** | **0.0000** | LR/SVM |  Dégénéré |

### Analyse comparée des méthodes

**Bag of Words (BoW) — la référence simple**

BoW représente chaque SMS par un vecteur de 8 475 dimensions, chaque dimension correspondant à un mot du vocabulaire. La valeur est simplement le nombre d'apparitions de ce mot dans le message. C'est une représentation creuse (sparse) : la plupart des cases valent zéro puisque chaque SMS n'utilise qu'une infime partie du vocabulaire total.

Le problème fondamental du BoW est qu'il traite tous les mots de manière égale : le mot "the" (apparaissant dans presque tous les messages, peu discriminant) a le même poids que "prize" (apparaissant presque uniquement dans les spams, très discriminant). Ce manque de pondération introduit du bruit dans la représentation.

**TF-IDF — la pondération intelligente**

TF-IDF corrige exactement ce défaut du BoW en multipliant deux facteurs :
- **TF (Term Frequency)** : fréquence du mot dans le document — récompense les mots importants dans ce message spécifique
- **IDF (Inverse Document Frequency)** : logarithme inverse du nombre de documents contenant ce mot — pénalise les mots apparaissant partout (peu informatifs) et valorise les mots rares (très informatifs)

Résultat : "prize", "winner", "free", "claim", "cash" obtiennent un score TF-IDF élevé dans les spams car ils y sont fréquents mais rares dans les ham. Le modèle apprend à s'appuyer sur ces mots discriminants. C'est ce qui explique le gain systématique de TF-IDF sur BoW (+0.0025 de F1).

**Word Embeddings — résultats décevants mais explicables**

Les trois méthodes d'embeddings (Word2Vec, GloVe, FastText) produisent des vecteurs denses de 100 dimensions qui sont supposés capturer la sémantique des mots. Deux mots utilisés dans des contextes similaires auront des vecteurs proches. L'idée est séduisante — mais les résultats sont inférieurs à TF-IDF sur ce dataset.

Pourquoi ? **Le corpus est trop petit.** Avec 5 574 SMS courts, la matrice de co-occurrence utilisée pour apprendre les embeddings est extrêmement clairsemée. Les vecteurs appris ne sont pas fiables — ils ne reflètent pas correctement les relations sémantiques entre mots.

**Le cas Word2Vec + LR/SVM (F1 = 0.000) est un cas de dégénérescence complète.** Une accuracy de 0.866 avec F1 = 0.000 signifie que le modèle prédit **systématiquement "ham"** pour tous les messages. Il n'a détecté aucun spam. Cela se produit quand les embeddings sont si mal estimés que les vecteurs des spams et des ham deviennent indiscernables dans l'espace 100D. Le SVM/LR ne trouve aucun hyperplan de séparation et classe tout dans la classe majoritaire (ham = 86.6 %).

GloVe et FastText évitent cette dégénérescence car ils sont construits différemment (TF-IDF+SVD et character n-grams) et capturent une information plus robuste, mais restent inférieurs à TF-IDF.

> **Conclusion vectorisation :** TF-IDF est la méthode optimale pour ce dataset. La pondération IDF est parfaitement alignée sur le problème de détection de spam : identifier les mots rares mais caractéristiques de la classe minoritaire. Les embeddings nécessitent des corpus bien plus grands (millions de documents) pour produire des vecteurs sémantiques fiables.

---

## 3.4 Impact du Modèle de Classification

### Résultats obtenus

| Modèle | Meilleur F1 | Vectorisation | Observation principale |
|---|---|---|---|
| **SVM** | **0.9368** | TF-IDF | Meilleur équilibre Precision/Recall |
| Logistic Regression | 0.9209 | BoW | Très proche du SVM |
| Random Forest | 0.8993 | TF-IDF | Precision=1.0 mais Recall faible |

### Analyse comparative des modèles

**SVM (Support Vector Machine) — le meilleur**

Le SVM est le modèle le plus performant, en particulier combiné avec TF-IDF. Son principe est de trouver l'hyperplan qui maximise la **marge de séparation** entre les classes spam et ham dans l'espace vectoriel de haute dimension. Ce mécanisme est particulièrement adapté aux données textuelles pour deux raisons :

1. Il gère très bien les espaces de haute dimension (8 475 dimensions pour TF-IDF) — là où d'autres modèles peinent à converger.
2. Il est robuste aux matrices creuses (sparse) — exactement ce que produisent BoW et TF-IDF.

Son F1-score de 0.9368 avec TF-IDF représente le meilleur équilibre entre precision (0.9852 — très peu de faux positifs) et recall (0.8929 — la grande majorité des spams est détectée).

**Logistic Regression — une excellente alternative**

La régression logistique obtient un F1 très proche du SVM (0.9209 avec BoW, soit un écart de seulement 0.016). Son avantage principal est l'**interprétabilité** : les coefficients du modèle indiquent directement quels mots poussent vers la classification "spam" et lesquels poussent vers "ham". On peut inspecter les 10 mots avec les plus grands coefficients positifs pour comprendre ce que le modèle a appris.

**Random Forest — comportement atypique**

Le Random Forest présente un comportement surprenant : avec TF-IDF, il atteint une **Precision de 1.000** (zéro faux positif — aucun ham classé comme spam) mais un Recall très faible (0.817). Cela signifie qu'il est ultra-conservateur : il ne déclare "spam" que lorsqu'il est absolument certain, laissant passer de nombreux vrais spams.

Ce comportement s'explique par la nature du Random Forest sur les données textuelles sparse : les arbres de décision individuels peinent à trouver des règles de découpage efficaces dans un espace de 8 475 dimensions dont la grande majorité est à zéro. L'agrégation favorise donc la classe majoritaire (ham).

> **Conclusion modèle :** SVM est le meilleur choix pour ce problème. La Logistic Regression est une alternative intéressante si l'interprétabilité est une priorité. Le Random Forest est à éviter sur les représentations textuelles sparse car il sacrifie le rappel (détection des spams) pour maximiser la précision.

---

## 3.5 Anomalie critique : Word2Vec dégénéré (F1 = 0.000)

Ce résultat mérite une attention particulière car il illustre un risque réel lorsqu'on utilise des embeddings sur de petits corpus.

**Ce qui s'est passé :** Word2Vec + LogisticRegression et Word2Vec + SVM affichent une accuracy de 0.866 avec F1 = 0.000, Precision = 0.000 et Recall = 0.000. Le modèle n'a détecté **aucun spam** — il a prédit "ham" pour chaque message du jeu de test.

**Pourquoi ?** Le processus d'apprentissage de Word2Vec nécessite d'observer chaque mot dans de nombreux contextes différents pour estimer ses relations sémantiques. Sur 5 574 SMS courts, cette condition n'est pas remplie. Les vecteurs produits sont essentiellement du bruit — ils ne capturent pas les relations réelles entre mots. En conséquence, les vecteurs moyens des spam et des ham occupent des zones similaires dans l'espace 100D : les deux classes sont indiscernables. Le SVM et la LR ne trouvent pas d'hyperplan de séparation et classent tout dans la classe majoritaire pour maximiser l'accuracy naïve (0.866 = proportion de ham dans le dataset).

**Solution :** Utiliser des embeddings **pré-entraînés** sur de grands corpus (GloVe entraîné sur 6 milliards de tokens de Wikipedia, FastText entraîné sur Common Crawl) au lieu d'entraîner from scratch. Ces vecteurs pré-appris sur des données massives donnent des représentations de qualité même quand le dataset cible est petit.


---
# 4. Conclusion Scientifique Argumentée

## 4.1 Validation des hypothèses

Ce TP a permis de tester quatre hypothèses fondamentales et d'en vérifier la validité expérimentale sur ce dataset :

| Hypothèse | Résultat | Preuve expérimentale |
|---|---|---|
| **H1 :** TF-IDF est meilleur que BoW |  **Validée** | F1 : 0.9368 (TF-IDF) vs 0.9343 (BoW) |
| **H2 :** Les embeddings surpassent TF-IDF |  **Invalidée** | F1 max embeddings : 0.8138 << 0.9368 (TF-IDF) |
| **H3 :** SVM > Logistic Regression pour le texte |  **Partiellement validée** | Écart faible : 0.9368 vs 0.9209 |
| **H4 :** Prétraitement avancé > Minimal |  **Invalidée** | Minimal=0.9471 > Stem=0.9398 > Lemma=0.9327 |

Les hypothèses H2 et H4 sont les plus importantes à commenter car elles vont à l'encontre des intuitions initiales.

**Sur H2 :** Les embeddings sont souvent présentés comme supérieurs à TF-IDF dans la littérature NLP — mais cette supériorité ne se manifeste que sur des corpus suffisamment grands et des tâches sémantiquement complexes. Sur 5 574 SMS courts à vocabulaire spam simple et répétitif, TF-IDF est non seulement meilleur mais aussi plus stable (aucune dégénérescence).

**Sur H4 :** Le stemming et la lemmatisation sont des outils puissants pour les textes longs et formels. Appliqués à des SMS spam, ils suppriment ou modifient précisément les tokens les plus discriminants. Le prétraitement optimal dépend toujours de la nature des données — il n'existe pas de pipeline universel.

---

## 4.2 Pipeline Optimal Recommandé

Sur la base de l'ensemble des 15 pipelines évalués, la combinaison optimale est :

> ## Nettoyage Minimal → TF-IDF → SVM linéaire
> **F1-score = 0.9471 | Precision = 0.9763 | Recall = 0.9196**

### Interprétation concrète des métriques

Pour comprendre ce que ces chiffres signifient en pratique, il faut les traduire en termes d'utilisation réelle :

**Precision = 0.9763** signifie que sur 100 alertes spam générées par le modèle, **97 sont de vrais spams** et seulement 3 sont des messages légitimes bloqués à tort. C'est un excellent résultat : les utilisateurs auront très rarement un message important bloqué par erreur.

**Recall = 0.9196** signifie que sur 100 vrais spams arrivant dans la boîte de réception, **92 sont détectés et bloqués**, et 8 passent inaperçus. Ce taux de détection est très satisfaisant pour une application réelle.

**F1-score = 0.9471** représente la moyenne harmonique de ces deux métriques. C'est le meilleur équilibre atteint sur ce dataset entre ne pas laisser passer de spams (rappel) et ne pas bloquer de messages légitimes (précision).

**Ce que les matrices de confusion confirment :**

Sur le jeu de test (1 672 messages) avec le pipeline optimal :
- **1 443 ham** sont correctement identifiés comme ham
- **5 ham** sont incorrectement marqués comme spam (faux positifs)
- **206 spam** sont correctement détectés et bloqués
- **18 spam** passent inaperçus (faux négatifs)

Ce profil d'erreurs est excellent pour une application de filtrage : il vaut toujours mieux laisser passer quelques spams que de bloquer des messages importants.

### Justification du choix

Ce pipeline est recommandé pour une mise en production car il combine plusieurs qualités essentielles :

**Performances :** Premier rang sur le F1-score parmi tous les pipelines testés. Le compromis precision/recall est optimal pour une application de filtrage de messages.

**Robustesse :** Aucune dégénérescence observée, contrairement à certains pipelines Word2Vec. Les résultats sont stables et prévisibles.

**Rapidité :** TF-IDF vectorise un nouveau SMS en quelques millisecondes. Le SVM prédit une classe en quelques microsecondes. Aucune ressource GPU n'est nécessaire.

**Maintenance :** Le modèle peut être mis à jour simplement avec de nouveaux exemples de spam sans reconstruire une architecture complexe. Lorsque de nouveaux types de spam émergent, il suffit d'ajouter des exemples étiquetés et de réentraîner.

**Interprétabilité :** Bien que le SVM soit moins directement interprétable que la Logistic Regression, on peut analyser les vecteurs de support et les mots à fort poids TF-IDF pour comprendre les décisions du modèle.

---

## 4.3 Classement des facteurs par importance

L'analyse de toutes les combinaisons permet de classer les quatre facteurs testés par leur impact sur le F1-score :

| Priorité | Facteur | Impact mesuré (ΔF1) | Explication |
|---|---|---|---|
|  **1er** | **Vectorisation** | Jusqu'à **0.937** | Word2Vec dégénéré (F1=0) vs TF-IDF (F1=0.937) |
|  **2ème** | **Modèle** | **0.118** | SVM (0.937) vs LR avec TF-IDF (0.819) |
|  **3ème** | **Nettoyage** | **0.014** | Minimal (0.947) vs Lemma (0.933) |
| 4ème | **Prétraitement** | **0.007** | Minimal (0.947) vs Stem (0.940) |

**La vectorisation est le facteur dominant** — et de très loin. Le choix de la méthode de représentation du texte conditionne plus les performances que le modèle lui-même, que le nettoyage ou que le prétraitement. Ce résultat est cohérent avec la littérature en NLP : la qualité de la représentation est fondamentale.

**Le modèle arrive en deuxième position.** Choisir SVM plutôt que LR avec TF-IDF apporte un gain de 0.118 de F1 — significatif. Le modèle de classification a un impact non négligeable mais reste secondaire par rapport à la vectorisation.

**Le nettoyage et le prétraitement ont un impact faible** (+/- 0.014 et 0.007). Ces étapes ont leur importance — surtout pour ne pas dégrader les performances — mais leur optimisation fine est peu rentable comparée au choix de la vectorisation.

---

## 4.4 Limites de ce travail

Ce TP présente plusieurs limites méthodologiques importantes à reconnaître honnêtement :

**Validation par un seul split (70/30) :** Les résultats obtenus dépendent de la graine aléatoire utilisée pour partager train et test. Un autre split pourrait donner des chiffres légèrement différents. Une **cross-validation à 10 folds** donnerait des estimations plus robustes avec des intervalles de confiance, permettant de dire que "TF-IDF+SVM donne un F1 de 0.937 ± 0.008" plutôt qu'un chiffre isolé.

**Déséquilibre de classes non traité :** L'option `class_weight='balanced'` du SVM et de la LR n'a pas été testée. Cette option augmente le poids des erreurs sur la classe minoritaire (spam) pendant l'entraînement et pourrait améliorer le recall.

**Dataset unique :** Les conclusions de ce TP sont valables pour ce dataset spécifique. Elles ne se généralisent pas nécessairement à d'autres datasets de spam (email spam, commentaires spam, etc.) dont les caractéristiques textuelles diffèrent.

---

## 4.5 Conclusion Générale

Ce TP illustre un principe fondamental en Machine Learning appliqué au traitement du langage naturel :

> **La complexité algorithmique n'est pas synonyme de performance. Elle doit être proportionnelle à la complexité réelle du problème et à la quantité de données disponibles.**

Sur ce dataset de 5 574 SMS courts au vocabulaire spam simple et répétitif, une pipeline classique — nettoyage minimal, TF-IDF, SVM linéaire — surpasse des approches modernes basées sur les embeddings. Ce résultat confirme les observations de la littérature : les méthodes dites "classiques" restent très compétitives sur les tâches de classification binaire à vocabulaire discriminant, et les embeddings n'apportent leur pleine valeur que sur des corpus volumineux et des tâches requérant une compréhension sémantique profonde.

Les résultats permettent également de formuler une règle pratique générale : **toujours commencer par les méthodes simples et n'augmenter la complexité que si les performances sont insuffisantes.** TF-IDF + SVM est simple, rapide, interprétable et atteint F1 = 0.9471 sur ce dataset. Passer à BERT ne ferait probablement gagner que 1 à 2 points de F1 supplémentaires — pour un coût en complexité et en ressources computationnelles sans commune mesure.

Enfin, ce TP souligne l'importance de **choisir les bonnes métriques d'évaluation.** L'accuracy seule aurait pu donner l'illusion que tous les pipelines sont bons (0.87 à 0.99). Le F1-score a révélé des comportements très différents : de l'excellent (TF-IDF + SVM, F1 = 0.9471) à la dégénérescence complète (Word2Vec + LR/SVM, F1 = 0.000). Adapter les métriques au problème — ici, le déséquilibre de classes et le coût asymétrique des erreurs — est une étape cruciale de toute démarche scientifique en Machine Learning.
